# 09 Statistics and tables
Subject-level tests with Holm correction for the confirmatory family (H1 to H3) and, separately, for the comparison with classical CSD, FDR for the rest, effect sizes with bootstrap intervals, summary tables. The figures are drawn in notebook 10.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
from retinopath import stats as S
n_boot = 1000 if DEV_MODE else cfg.stats.n_boot
reference = pd.read_csv(RES / "h1_reference.csv"); classical = pd.read_csv(RES / "protocols.csv"); learned = pd.read_csv(RES / "learned.csv")
learned["variant"] = learned.method.str.extract(r"net_(.*)_s\d+")[0]
clinical = [protocol for protocol in cfg.clinical_protocols if protocol in cfg.run_protocols]; confirmatory = []

# H1: RSI > 0 for each area pair and each retinotopic axis, reference data
ref_rows = reference[reference.method == "msmt_csd"]
for pair in ("V1-V2", "V2-V3", "V1-V3"):
    for quantity in ("dlogecc", "dang"):
        confirmatory.append(dict(hyp="H1", test=f"RSI {quantity} {pair} > 0", **S.paired_test(ref_rows[f"rsi_{quantity}_{pair}"], alternative="greater", n_boot=n_boot)))

# H3: proposed (path loss) vs the same network with voxel loss only; seeds averaged within subject, all seeds required
all_seeds = lambda values: values.mean(skipna=False)
for protocol in clinical:
    wide = learned[learned.protocol == protocol].groupby(["sid", "variant"])[["rsi_ree_all", "acc"]].agg(all_seeds).unstack("variant")
    if {"path", "voxel"} <= set(wide["rsi_ree_all"].columns):
        confirmatory.append(dict(hyp="H3", test=f"RSI path > voxel-only, {protocol}", **S.paired_test(wide["rsi_ree_all"]["path"], wide["rsi_ree_all"]["voxel"], "greater", n_boot=n_boot)))
        confirmatory.append(dict(hyp="H3", test=f"ACC path not worse than voxel-only, {protocol}", **S.paired_test(wide["acc"]["path"], wide["acc"]["voxel"], "greater", shift=cfg.stats.acc_margin, n_boot=n_boot)))

# H2: the voxel-only network recovers more of the lost voxel accuracy than of the lost RSI.
# recovery = (network - classical CSD on the same protocol) / (reference - classical CSD); reference ACC is 1 by definition.
csd_scores = classical[classical.method == "csd"].set_index(["sid", "protocol"]); ref_rsi = ref_rows.set_index("sid")["rsi_ree_all"]
voxel_scores = learned[learned.variant == "voxel"].groupby(["sid", "protocol"])[["acc", "rsi_ree_all"]].agg(all_seeds)
joined = voxel_scores.join(csd_scores[["acc", "rsi_ree_all"]], rsuffix="_csd", how="inner").reset_index(); joined["rsi_ref"] = joined.sid.map(ref_rsi)
joined["rec_acc"] = (joined.acc - joined.acc_csd) / (1 - joined.acc_csd); rsi_lost = joined.rsi_ref - joined.rsi_ree_all_csd
joined["rec_rsi"] = ((joined.rsi_ree_all - joined.rsi_ree_all_csd) / rsi_lost).where(rsi_lost >= 0.02)      # under 0.02 there is nothing to recover
joined = joined[joined.protocol.isin(clinical)]; print("subject x protocol cells without a loss of RSI (left out of H2):", int(joined.rec_rsi.isna().sum()), "of", len(joined))
joined = joined.replace([np.inf, -np.inf], np.nan); both = joined.rec_acc.notna() & joined.rec_rsi.notna()
joined[["rec_acc", "rec_rsi"]] = joined[["rec_acc", "rec_rsi"]].where(both)   # same cells for both recoveries
recovery = joined.groupby("sid")[["rec_acc", "rec_rsi"]].median()
confirmatory.append(dict(hyp="H2", test="recovery of ACC > recovery of RSI, voxel-only network", **S.paired_test(recovery.rec_acc, recovery.rec_rsi, "greater", n_boot=n_boot)))

confirmatory = pd.DataFrame(confirmatory); confirmatory["p_holm"] = S.correct(confirmatory.p, "holm"); confirmatory["significant"] = confirmatory.p_holm < cfg.stats.alpha; confirmatory.to_csv(RES / "confirmatory_tests.csv", index=False)
print(len(confirmatory), "confirmatory tests,", int(confirmatory.p.isna().sum()), "without a p-value")
confirmatory.round(4)

In [ ]:
# Comparison with classical CSD on each clinical-grade protocol (pre-specified, Holm within this family):
# the voxel-only network lowers RSI; the path network does not (margin stats.rsi_margin), has higher ACC and
# follows the reference paths more closely than classical CSD and than the voxel-only network.
network = learned.groupby(["sid", "protocol", "variant"])[["rsi_ree_all", "acc"]].agg(all_seeds); rows = []
path_error = pd.read_csv(RES / "path_error.csv"); path_error["variant"] = path_error.method.str.extract(r"net_(.*)_s\d+")[0].fillna(path_error.method)
path_error = path_error.groupby(["sid", "protocol", "variant"]).path_err_mm.agg(all_seeds).unstack("variant")
for protocol in clinical:
    baseline = csd_scores.xs(protocol, level="protocol")[["rsi_ree_all", "acc"]]; drift = path_error.xs(protocol, level="protocol")
    of = lambda variant: network.xs((protocol, variant), level=("protocol", "variant")).reindex(baseline.index)
    if {"path", "voxel"} <= set(network.index.get_level_values("variant")):
        path_scores, voxel_only = of("path"), of("voxel")
        tests = [("RSI voxel-only < classical CSD", voxel_only.rsi_ree_all, baseline.rsi_ree_all, "less", 0.0),
                 ("RSI path not worse than classical CSD", path_scores.rsi_ree_all, baseline.rsi_ree_all, "greater", cfg.stats.rsi_margin),
                 ("ACC path > classical CSD", path_scores.acc, baseline.acc, "greater", 0.0),
                 ("path error path < classical CSD", drift["path"], drift["csd"], "less", 0.0),
                 ("path error path < voxel-only", drift["path"], drift["voxel"], "less", 0.0)]
        rows += [dict(protocol=protocol, test=test, **S.paired_test(first, second, alternative, shift=shift, n_boot=n_boot)) for test, first, second, alternative, shift in tests]
against_csd = pd.DataFrame(rows)
if len(against_csd): against_csd["p_holm"] = S.correct(against_csd.p, "holm"); against_csd["significant"] = against_csd.p_holm < cfg.stats.alpha; against_csd.to_csv(RES / "csd_comparison_tests.csv", index=False)
against_csd.round(4) if len(against_csd) else print("no network scores")

In [ ]:
# Training size: the H3 tests and the comparison with classical CSD of this run and of every run in
# training_size_runs (same experiment with other n_train) that has both tables.
from retinopath.core import ROOT
runs = {int(n_train): ROOT / folder / ("dev" if cfg.dev_mode else "") / "results" for n_train, folder in cfg.get("training_size_runs", {}).items()}
runs[cfg.n_train] = RES; by_size = []
for n_train, folder in sorted(runs.items()):
    if not ((folder / "confirmatory_tests.csv").exists() and (folder / "csd_comparison_tests.csv").exists()): continue
    h3 = pd.read_csv(folder / "confirmatory_tests.csv").query("hyp == 'H3'")
    h3 = h3.assign(protocol=h3.test.str.rsplit(", ", n=1).str[-1])
    by_size += [h3.assign(family="confirmatory", n_train=n_train), pd.read_csv(folder / "csd_comparison_tests.csv").assign(family="csd_comparison", n_train=n_train)]
training_size = pd.concat(by_size, ignore_index=True).sort_values(["family", "test", "protocol", "n_train"]) if by_size else pd.DataFrame()
if len(training_size): training_size.to_csv(RES / "training_size_tests.csv", index=False)
print("training sizes found:", sorted(map(int, training_size.n_train.unique())) if len(training_size) else "none")
training_size.round(4) if len(training_size) else None

In [ ]:
# Exploratory grid: every method and protocol against classical CSD on the same protocol (FDR).
# Networks are averaged over training seeds first, so each variant is one method.
combined = pd.concat([classical, learned.assign(method="net_" + learned.variant)]).groupby(["sid", "protocol", "method"], as_index=False)["rsi_ree_all"].mean(); rows = []
for (protocol, method), group_scores in combined.groupby(["protocol", "method"]):
    baseline = combined[(combined.protocol == protocol) & (combined.method == "csd")].set_index("sid")["rsi_ree_all"]
    if method != "csd" and len(baseline):
        values = group_scores.set_index("sid")["rsi_ree_all"].reindex(baseline.index)
        rows.append(dict(protocol=protocol, method=method, **S.paired_test(values, baseline, n_boot=n_boot)))
exploratory = pd.DataFrame(rows)
if len(exploratory): exploratory["p_fdr"] = S.correct(exploratory.p, "fdr_bh"); exploratory.to_csv(RES / "exploratory_tests.csv", index=False)
# Mixed model on the balanced part of the grid: clinical-grade protocols x the methods run on all of them
balanced = combined[combined.protocol.isin(clinical)]; protocols_per_method = balanced.groupby("method").protocol.nunique(); balanced = balanced[balanced.method.isin(protocols_per_method[protocols_per_method == len(clinical)].index)]
mixed = S.mixed_model(balanced.rename(columns={"rsi_ree_all": "rsi"}))
if mixed is not None: mixed.to_csv(RES / "mixed_model.csv"); display(mixed)
exploratory.round(4) if len(exploratory) else print("no exploratory comparisons")

In [ ]:
# Pairwise RSI comparisons between all methods on each clinical-grade protocol: two-sided Wilcoxon, FDR over every
# pair of every protocol (exploratory). Networks are averaged over training seeds first, so each variant is one method.
combined = pd.concat([classical, learned.assign(method="net_" + learned.variant)])
combined = combined[combined.protocol.isin(clinical)].groupby(["sid", "protocol", "method"]).rsi_ree_all.mean().unstack("method"); rows = []
for protocol, group_scores in combined.groupby(level="protocol"):
    methods = [method for method in group_scores.columns if group_scores[method].notna().any()]
    for index, first in enumerate(methods):
        for second in methods[index + 1:]:
            rows.append(dict(protocol=protocol, method_1=first, method_2=second, **S.paired_test(group_scores[first], group_scores[second], n_boot=n_boot)))
pairwise = pd.DataFrame(rows)
if len(pairwise): pairwise["p_fdr"] = S.correct(pairwise.p, "fdr_bh"); pairwise.to_csv(RES / "pairwise_tests.csv", index=False)
print(len(pairwise), "pairs,", int((pairwise.p_fdr < cfg.stats.alpha).sum()) if len(pairwise) else 0, "with FDR p <", cfg.stats.alpha)

In [ ]:
# The path vs voxel-only contrast scored with the two keys that need no functional scan of the subject (descriptive)
alternative = []
for protocol in clinical:
    for column, key in (("template_rsi_ree_all", "group-average pRF"), ("atlas_rsi_ree_all", "anatomical atlas")):
        wide = learned[learned.protocol == protocol].pivot_table(index="sid", columns="variant", values=column, aggfunc="mean")
        if {"path", "voxel"} <= set(wide.columns):
            alternative.append(dict(protocol=protocol, key=key, **S.paired_test(wide["path"], wide["voxel"], n_boot=n_boot)))
alternative = pd.DataFrame(alternative); alternative.to_csv(RES / "alternative_key_tests.csv", index=False)
# agreement of the atlas key with the measured key over all subjects, protocols and methods
rsi_pairs = pd.concat([reference, classical, learned])[["rsi_ree_all", "atlas_rsi_ree_all"]].dropna()
print("Spearman r, measured-key RSI vs atlas-key RSI:", round(rsi_pairs.corr(method="spearman").iloc[0, 1], 3), "| n =", len(rsi_pairs)); alternative.round(4)

In [ ]:
# Sampling threshold and lobe width (descriptive): the networks against each other and against classical CSD when the
# same fODFs are tracked with DIPY's default sampling threshold, and the width of their fODF lobes (notebook 08).
rows = []
if (RES / "sampling.csv").exists():
    sampling = pd.read_csv(RES / "sampling.csv"); sampling["variant"] = sampling.method.str.extract(r"net_(.*)_s\d+")[0].fillna(sampling.method)
    sampling = sampling.groupby(["sid", "protocol", "variant"])[["rsi_ree_all", "lobe_width"]].agg(all_seeds)
    for protocol in clinical:
        for column, label in (("rsi_ree_all", "RSI at sampling threshold 0.1"), ("lobe_width", "lobe width")):
            wide = sampling[column].xs(protocol, level="protocol").unstack("variant")
            rows += [dict(protocol=protocol, test=f"{label}, {first} - {second}", **S.paired_test(wide[first], wide[second], n_boot=n_boot))
                     for first, second in (("path", "voxel"), ("path", "csd"), ("voxel", "csd")) if {first, second} <= set(wide.columns)]
sampling_tests = pd.DataFrame(rows)
if len(sampling_tests): sampling_tests.to_csv(RES / "sampling_tests.csv", index=False)
sampling_tests.round(4) if len(sampling_tests) else print("no sampling table: run notebook 08 first")

In [ ]:
# Main table: median and inter-quartile range over subjects
summary = lambda series: f"{series.median():.3f} [{series.quantile(.25):.3f}, {series.quantile(.75):.3f}]" if series.notna().any() else "not available"
combined = pd.concat([reference[reference.method != "proximity"], classical, learned])
columns = {"rsi_ree_all": "RSI", "atlas_rsi_ree_all": "RSI, atlas key", "ree_median_all": "endpoint error (deg)", "acc": "ACC", "peak_err_deg": "peak error (deg)", "wdice_ref": "weighted Dice", "n_pairs": "V1-V3 streamlines"}
main_table = combined.groupby(["protocol", "method"])[list(columns)].agg(summary).rename(columns=columns)
main_table.to_csv(RES / "main_results.csv")
run_log = pd.read_csv(cfg.P.out / "run_log.csv"); print("hours logged so far:", round(run_log.hours.sum(), 2)); main_table

In [ ]:
# External data: HCP 7T scans and NSD. Reported per method with medians; with 8 NSD subjects the paired comparison
# is shown with its interval and is descriptive, not part of the confirmatory family.
external, rows = [], []
for dataset, file in (("HCP 7T", "external_7t.csv"), ("NSD", "external_nsd.csv")):
    if (RES / file).exists():
        group_scores = pd.read_csv(RES / file); group_scores["dataset"] = dataset; group_scores["variant"] = group_scores.method.str.extract(r"net_(.*)_s\d+")[0].fillna(group_scores.method); external.append(group_scores)
        wide = group_scores.pivot_table(index="sid", columns="variant", values="rsi_ree_all", aggfunc="mean")
        if {"path", "voxel"} <= set(wide.columns):
            rows.append(dict(dataset=dataset, test="RSI path - voxel-only", **S.paired_test(wide["path"], wide["voxel"], n_boot=n_boot)))
        if "csd" in wide.columns and "path" in wide.columns:
            rows.append(dict(dataset=dataset, test="RSI path - classical CSD", **S.paired_test(wide["path"], wide["csd"], n_boot=n_boot)))
if external:
    external = pd.concat(external); external_columns = [column_name for column_name in ("rsi_ree_all", "rsi_ree_geodesic_all", "template_rsi_ree_all", "atlas_rsi_ree_all", "ree_median_all", "n_pairs") if column_name in external]
    external_table = external.groupby(["dataset", "variant"])[external_columns].agg(summary); external_table.to_csv(RES / "external_results.csv")
    pd.DataFrame(rows).to_csv(RES / "external_tests.csv", index=False); display(external_table); display(pd.DataFrame(rows).round(4))
reproducibility = [pd.read_csv(RES / file).assign(dataset=dataset_name) for dataset_name, file in (("HCP retest", "retest_overlap.csv"), ("NSD run 1 vs 2", "nsd_run_overlap.csv")) if (RES / file).exists()]
if reproducibility:
    reproducibility_table = pd.concat(reproducibility).groupby("dataset")[["wdice_retest", "badj_retest_mm", "vertex_count_r"]].agg(summary); reproducibility_table.to_csv(RES / "reproducibility.csv"); display(reproducibility_table)